# RC-Car Person Detector V1 Training
커널을 **RC Person Detector**로 선택한 뒤 위에서 아래로 실행합니다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs' / 'train_v1.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs' / 'train_v1.json').exists(), '프로젝트 루트를 찾지 못했습니다.'
print('ROOT:', ROOT)
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 1024**3, 2))

In [ ]:
assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다. 전체 학습을 시작하지 마세요.'
config = json.loads((ROOT / 'configs' / 'train_v1.json').read_text(encoding='utf-8'))
config

## 사전 검사
전체 학습 전에 training step, inference, CUDA mini-overfit 검사를 실행합니다.

In [ ]:
tests = [
    [sys.executable, str(ROOT / 'scripts' / '13_test_training_step.py'), '--root', str(ROOT)],
    [sys.executable, str(ROOT / 'scripts' / '14_test_inference.py'), '--root', str(ROOT)],
    [sys.executable, str(ROOT / 'scripts' / '15_mini_overfit.py'), '--root', str(ROOT), '--device', 'cuda'],
]
for command in tests:
    subprocess.run(command, cwd=ROOT, check=True)

## 전체 학습
VRAM에 맞게 `BATCH_SIZE`를 변경할 수 있습니다. 기본값은 8입니다.

In [ ]:
BATCH_SIZE = 8
EPOCHS = 100
command = [
    sys.executable, str(ROOT / 'scripts' / '16_train.py'),
    '--root', str(ROOT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
]
subprocess.run(command, cwd=ROOT, check=True)

## 중단된 학습 재개

In [ ]:
last_checkpoint = ROOT / 'results' / 'training' / 'person_detector_v1' / 'checkpoints' / 'last.pt'
print(last_checkpoint, last_checkpoint.exists())
# 실행하려면 다음 두 줄의 주석을 해제하세요.
# command = [sys.executable, str(ROOT / 'scripts' / '16_train.py'), '--root', str(ROOT), '--resume', str(last_checkpoint)]
# subprocess.run(command, cwd=ROOT, check=True)

## Loss 그래프

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
history_path = ROOT / 'results' / 'training' / 'person_detector_v1' / 'history.csv'
history = pd.read_csv(history_path)
ax = history.plot(x='epoch', y=['train_total', 'valid_total'], grid=True, figsize=(10, 5))
ax.set_ylabel('Loss')
plt.show()
history.tail()